In [111]:
import pandas as pd
import networkx as nx

filename = 'Edges_2015_All_P_Capacity_Distance.csv'
edges_distance_df = pd.read_csv(filename)
# tuple_list = list(edges_distance_df[['id1', 'id2','weight-GC']].itertuples(index=False, name=None))
# G = nx.Graph()
# G.add_weighted_edges_from(tuple_list)
tuple_list = list(edges_distance_df[['id1', 'id2']].itertuples(index=False, name=None))
G = nx.Graph()
G.add_edges_from(tuple_list)

数据详见论文'Modular gateway-ness connectivity and structural core organization in maritime network science'附录

In [113]:
print('网络的节点数为:', G.number_of_nodes())
print('网络的边数为:', G.number_of_edges())
print('网络的平均度为:', G.number_of_edges() * 2 / (G.number_of_nodes()))
print('网络的集聚系数为:', nx.average_clustering(G))
print('网络的密度为:', nx.density(G))
print('网络的直径为:', nx.diameter(G))
print('网络的平均最短路径为:', nx.average_shortest_path_length(G))

网络的节点数为: 977
网络的边数为: 16680
网络的平均度为: 34.1453428863869
网络的集聚系数为: 0.7130511856826472
网络的密度为: 0.034984982465560345
网络的直径为: 5
网络的平均最短路径为: 2.6712712049264224


# 无向加权网络

In [155]:
import pandas as pd
import networkx as nx

filename = 'Edges_2015_All_P_Capacity_Distance.csv'
edges_distance_df = pd.read_csv(filename)
tuple_list = list(edges_distance_df[['id1', 'id2','weight-GC']].itertuples(index=False, name=None))
G = nx.Graph()
G.add_weighted_edges_from(tuple_list)

# 无向无权网络

In [161]:
import pandas as pd
import networkx as nx

filename = 'Edges_2015_All_P_Capacity_Distance.csv'
edges_distance_df = pd.read_csv(filename)
tuple_list = list(edges_distance_df[['id1', 'id2']].itertuples(index=False, name=None))
G = nx.Graph()
G.add_edges_from(tuple_list)

# GN算法

## 无权

In [77]:
import networkx as nx
from networkx.algorithms.community import girvan_newman
from networkx.algorithms.community.quality import modularity

# GN算法
def girvan_newman_best_partition(G):
    max_modularity = float('-inf')
    best_partition = None
    
    # 使用 girvan_newman 函数获取按模块度递减排序的社区划分序列
    comp = girvan_newman(G)
    for communities in comp:
        communities = list(communities)  # 转换生成器为列表
        modularity_value = modularity(G, communities)
        if modularity_value > max_modularity:
            max_modularity = modularity_value
            best_partition = communities
    
    return best_partition, max_modularity


gn_partition, gn_modularity = girvan_newman_best_partition(G)
# 将社区划分转换为字典格式，社区编号从0开始
gn_partition = {node: community_id for community_id, community in enumerate(gn_partition) for node in community}
# 打印 GN 算法找到的最大模块度和社区数量
print(f"GN Modularity: {gn_modularity}")
print(f"Number of communities: {len(gn_partition)}")

KeyboardInterrupt: 

# FN算法

## 无权

In [162]:
fn_communities = list(nx.algorithms.community.greedy_modularity_communities(G))
# fn_communities = list(nx.algorithms.community.greedy_modularity_communities(G,weight='weight'))
fn_modularity = modularity(G, fn_communities)
fn_partition = {node: i for i, comm in enumerate(fn_communities) for node in comm}
print(f"FN Modularity: {fn_modularity}")
num_communities = len(set(fn_partition.values()))
print(f"Number of communities: {num_communities}")

FN Modularity: 0.41267538666908193
Number of communities: 6


## 加权

In [139]:
# fn_communities = list(nx.algorithms.community.greedy_modularity_communities(G))
fn_communities = list(nx.algorithms.community.greedy_modularity_communities(G,weight='weight'))
fn_modularity = modularity(G, fn_communities)
fn_partition = {node: i for i, comm in enumerate(fn_communities) for node in comm}
print(f"FN Modularity: {fn_modularity}")
num_communities = len(set(fn_partition.values()))
print(f"Number of communities: {num_communities}")

FN Modularity: 0.764616855543021
Number of communities: 19


# Louvain算法

## 无权

In [179]:
import networkx.algorithms.community as nx_comm
louvain_communities = list(nx_comm.louvain_communities(G))
louvain_modularity = modularity(G, louvain_communities)
louvain_partition = {node: i for i, comm in enumerate(louvain_communities) for node in comm}
print(f"Louvain Modularity: {louvain_modularity}")
num_communities = len(louvain_communities)
print(f"Number of communities: {num_communities}")

Louvain Modularity: 0.47360316782545187
Number of communities: 8


## 加权

In [140]:
import networkx.algorithms.community as nx_comm
louvain_communities = list(nx_comm.louvain_communities(G))
louvain_modularity = modularity(G, louvain_communities)
louvain_partition = {node: i for i, comm in enumerate(louvain_communities) for node in comm}
print(f"Louvain Modularity: {louvain_modularity}")
num_communities = len(louvain_communities)
print(f"Number of communities: {num_communities}")

Louvain Modularity: 0.7651289369859424
Number of communities: 18


# Infomap算法

## 无权

In [165]:
from infomap import Infomap

# Infomap算法
def infomap_community_detection(G):
    infomap = Infomap()
    for e in G.edges():
        infomap.add_link(*e)
    infomap.run()
    infomap_partition = {node.node_id: node.module_id for node in infomap.nodes}
    infomap_codelength = infomap.codelength
    return infomap_partition,infomap_codelength

infomap_partition,infomap_codelength = infomap_community_detection(G)
infomap_communities = {}
for node, community in infomap_partition.items():
    if community not in infomap_communities:
        infomap_communities[community] = []
    infomap_communities[community].append(node)
infomap_modularity = modularity(G, list(infomap_communities.values()))
print(f"Infomap Modularity: {infomap_modularity}")
print(f"Infomap Codelength: {infomap_codelength}")
num_communities = len(set(infomap_partition.values()))
print(f"Number of communities: {num_communities}")

Infomap Modularity: 0.4102307758110059
Infomap Codelength: 8.552664916265288
Number of communities: 26


## 加权

In [141]:
from infomap import Infomap

# Infomap算法
def infomap_community_detection(G):
    infomap = Infomap()
    # 遍历边，加入权重信息
    for u, v, data in G.edges(data=True):
        weight = data.get('weight', 1.0)  # 如果没有权重属性，默认为1.0
        infomap.add_link(u, v, weight)
    infomap.run()
    infomap_partition = {node.node_id: node.module_id for node in infomap.nodes}
    infomap_codelength = infomap.codelength
    return infomap_partition, infomap_codelength

infomap_partition, infomap_codelength = infomap_community_detection(G)
infomap_communities = {}
for node, community in infomap_partition.items():
    if community not in infomap_communities:
        infomap_communities[community] = []
    infomap_communities[community].append(node)

infomap_modularity = modularity(G, list(infomap_communities.values()))
print(f"Infomap Modularity: {infomap_modularity}")
print(f"Infomap Codelength: {infomap_codelength}")
num_communities = len(set(infomap_partition.values()))
print(f"Number of communities: {num_communities}")

Infomap Modularity: 0.5980590800407024
Infomap Codelength: 3.9919947763056274
Number of communities: 7


# LPA算法

## 无权

In [166]:
from networkx.algorithms.community import label_propagation_communities
label_propagation_communities_list = list(label_propagation_communities(G))
label_propagation_partition = {node: i for i, community in enumerate(label_propagation_communities_list) for node in community}
label_propagation_modularity = modularity(G, label_propagation_communities_list)
print(f"LPA Modularity: {label_propagation_modularity}")
num_communities = len(set(label_propagation_partition.values()))
print(f"Number of communities: {num_communities}")

LPA Modularity: 0.1012822242206234
Number of communities: 13


# 谱聚类算法

## 无权

In [167]:
from sklearn.cluster import SpectralClustering

# 谱聚类算法
def spectral_clustering_community_detection(G, n_clusters):
    adjacency_matrix = nx.to_numpy_matrix(G)
    sc = SpectralClustering(n_clusters=n_clusters, affinity='precomputed', n_init=100)
    sc.fit(adjacency_matrix)
    spectral_partition = {list(G.nodes())[i]: sc.labels_[i] for i in range(len(G.nodes()))}
    return spectral_partition

n_clusters = len(louvain_communities)
spectral_partition = spectral_clustering_community_detection(G, n_clusters)
spectral_communities = {}
for node, community in spectral_partition.items():
    if community not in spectral_communities:
        spectral_communities[community] = []
    spectral_communities[community].append(node)
spectral_modularity = modularity(G, list(spectral_communities.values()))
print(f"谱聚类 Modularity: {spectral_modularity}")
num_communities = len(set(spectral_partition.values()))
print(f"Number of communities: {num_communities}")

谱聚类 Modularity: 0.35197200872395606
Number of communities: 9


## 加权

In [157]:
import networkx as nx
from sklearn.cluster import SpectralClustering

# 谱聚类算法
def spectral_clustering_community_detection(G, n_clusters):
    # 转换为加权邻接矩阵
    adjacency_matrix = nx.to_numpy_array(G, weight='weight')
    # 使用预计算的相似性矩阵进行谱聚类
    sc = SpectralClustering(n_clusters=n_clusters, affinity='precomputed', n_init=100)
    sc.fit(adjacency_matrix)
    # 将聚类结果存储在字典中
    spectral_partition = {list(G.nodes())[i]: sc.labels_[i] for i in range(len(G.nodes()))}
    return spectral_partition

n_clusters = len(louvain_communities)
spectral_partition = spectral_clustering_community_detection(G, n_clusters)
spectral_communities = {}
for node, community in spectral_partition.items():
    if community not in spectral_communities:
        spectral_communities[community] = []
    spectral_communities[community].append(node)
spectral_modularity = modularity(G, list(spectral_communities.values()))
print(f"谱聚类 Modularity: {spectral_modularity}")
num_communities = len(set(spectral_partition.values()))
print(f"Number of communities: {num_communities}")

谱聚类 Modularity: 0.20188719421668577
Number of communities: 18


# 汇总结果

In [172]:
file_name = 'GLSN_'
# 输出每个算法的划分结果
results = pd.DataFrame({'ID': list(G.nodes())})
# results['gn'] = results['ID'].map(gn_partition)
results['fn'] = results['ID'].map(fn_partition)
results['infomap'] = results['ID'].map(infomap_partition)
results['label_propagation'] = results['ID'].map(label_propagation_partition)
results['louvain'] = results['ID'].map(louvain_partition)
results['spectral'] = results['ID'].map(spectral_partition)

# # 输出模块度值和割值
# metrics = {
#     'algorithm': ['GN', 'fn', 'Infomap', 'Label Propagation', 'Spectral Clustering', 'Louvain'],
#     'modularity': [gn_modularity, fn_modularity, infomap_modularity, label_propagation_modularity, spectral_modularity, louvain_modularity]
# }

# 输出模块度值和割值,不含GN
metrics = {
    'algorithm': [ 'fn', 'Louvain', 'Infomap', 'Label Propagation', 'Spectral Clustering'],
    'modularity': [ fn_modularity, louvain_modularity, infomap_modularity, label_propagation_modularity, spectral_modularity]
}
metrics_df = pd.DataFrame(metrics)

# 保存结果
results.to_csv(file_name + 'community_detection_results.csv', index=False)
metrics_df.to_csv(file_name + 'community_detection_metrics.csv', index=False)

print("社区划分结果已保存为 'community_detection_results.csv'")
print("算法评价指标已保存为 'community_detection_metrics.csv'")

社区划分结果已保存为 'community_detection_results.csv'
算法评价指标已保存为 'community_detection_metrics.csv'


In [180]:
file_name = 'GLSN_'

# 输出每个算法的划分结果
# gn_results = pd.DataFrame({'ID': list(G.nodes()), 'gn_community': list(map(gn_partition.get, G.nodes()))})
fn_results = pd.DataFrame({'ID': list(G.nodes()), 'fn_community': list(map(fn_partition.get, G.nodes()))})
infomap_results = pd.DataFrame({'ID': list(G.nodes()), 'infomap_community': list(map(infomap_partition.get, G.nodes()))})
label_propagation_results = pd.DataFrame({'ID': list(G.nodes()), 'label_propagation_community': list(map(label_propagation_partition.get, G.nodes()))})
spectral_results = pd.DataFrame({'ID': list(G.nodes()), 'spectral_community': list(map(spectral_partition.get, G.nodes()))})

# 计算 Louvain 社区划分结果
# louvain_partition = {node: i for i, comm in enumerate(louvain_communities) for node in comm}
louvain_results = pd.DataFrame({'ID': list(G.nodes()), 'louvain_community': list(map(louvain_partition.get, G.nodes()))})

# 保存每个算法的结果
# gn_results.to_csv(file_name+'gn_results.csv', index=False)
fn_results.to_csv(file_name+'fn_results.csv', index=False)
infomap_results.to_csv(file_name+'infomap_results.csv', index=False)
label_propagation_results.to_csv(file_name+'label_propagation_results.csv', index=False)
spectral_results.to_csv(file_name+'spectral_results.csv', index=False)
louvain_results.to_csv(file_name+'louvain_results.csv', index=False)

print("所有算法的社区划分结果已保存为 CSV 文件")


所有算法的社区划分结果已保存为 CSV 文件
